In [1]:
from pathlib import Path
import sys
import numpy as np
PROJETO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))

from mvnview import *

In [2]:
PASTA_ARQUIVOS_MVNX = PROJETO / "capture_data" / "raw_mvnx" / "files"
PASTA_ARQUIVOS_NPZ = PROJETO / "capture_data" / "converted_npz" / "files"
PASTA_ARQUIVOS_NPZ.mkdir(parents=True, exist_ok=True)

NOME_ARQUIVO_NPZ = "captura_de_movimento_convertida.npz"
arquivo_npz_convertido = PASTA_ARQUIVOS_NPZ / NOME_ARQUIVO_NPZ
data = MocapData(arquivo_npz_convertido)

In [3]:
def better_orientation(data, target_name):
    if data.segment_names is None:
        print("None")
        return
    names = []
    for name in data.segment_names:
        names.append(str(name))

    frame_num = data.orientation.shape[0]
    segment_num = len(names)

    reshaped_ori = data.orientation.reshape(frame_num, segment_num, 4)

    better_orientation_dict = {}
    for i, name in enumerate(names):
        better_orientation_dict[name] = reshaped_ori[:, i, :]
        
    # quat manipulation
    
    quat = better_orientation_dict[target_name]
    # acess individual components in all frames
    q0 = quat[:,0] # scalar
    q1 = quat[:,1] # vector x
    q2 = quat[:,2] # vector y
    q3 = quat[:,3] # vector z

    matrix_dcm = create_directional_cosine_matrix(q0,q1,q2,q3)
    return quat, matrix_dcm

In [4]:
def create_directional_cosine_matrix(q0,q1,q2,q3):
    # math derived from "Euler and Quaternion Angles: Differences and Why it Matters" by Internal Labs 
    q0_2 = q0**2
    q1_2 = q1**2
    q2_2 = q2**2
    q3_2 = q3**2

    c11 = q0_2 + q1_2 - q2_2 - q3_2
    c12 = 2 * (q1 * q2 - q0 * q3)
    c13 = 2 * (q1 * q3 + q0 * q2)

    c21 = 2 * (q1 * q2 + q0 * q3)
    c22 = q0_2 - q1_2 + q2_2 - q3_2
    c23 = 2 * (q2 * q3 - q0 * q1)

    c31 = 2 * (q1 * q3 - q0 * q2)
    c32 = 2 * (q2 * q3 + q0 * q1)
    c33 = q0_2 - q1_2 - q2_2 + q3_2

    matrix_dcm = np.stack([
        np.stack([c11, c12, c13], axis=-1),
        np.stack([c21, c22, c23], axis=-1),
        np.stack([c31, c32, c33], axis=-1)
    ], axis=-2)

    return matrix_dcm

In [5]:
ori0 = better_orientation(data,'RightShoulder')[0]
print(ori0)

orio1 = better_orientation(data,'RightShoulder')[1]
print(orio1)

[[ 0.98888  -0.010079  0.148373 -0.      ]
 [ 0.989074 -0.013479  0.146748 -0.004098]
 [ 0.989236 -0.016879  0.145119 -0.008196]
 ...
 [ 0.46721  -0.0028    0.177362  0.86617 ]
 [ 0.47759  -0.004032  0.176062  0.860752]
 [-0.487903  0.005264 -0.174737 -0.855213]]
[[[ 0.95597069 -0.0029909   0.29344618]
  [-0.0029909   0.99979662  0.01993384]
  [-0.29344618 -0.01993384  0.95576752]]

 [[ 0.95689729  0.00415042  0.29039974]
  [-0.01206248  0.99960388  0.02546071]
  [-0.29017879 -0.0278662   0.95656751]]

 [[ 0.95774607  0.01131663  0.28739056]
  [-0.02111448  0.99929531  0.03101584]
  [-0.2868372  -0.03577342  0.95731061]]

 ...

 [[-0.56341472 -0.8103598   0.16088005]
  [ 0.80837334 -0.50051585  0.30986766]
  [-0.17058115  0.30463491  0.93707053]]

 [[-0.54378337 -0.82359286  0.1612298 ]
  [ 0.82075333 -0.48182023  0.30694272]
  [-0.17511201  0.29924015  0.93797213]]

 [[-0.52384525 -0.83636161  0.16150573]
  [ 0.83268235 -0.46283463  0.30401135]
  [-0.1795131   0.29373807  0.93887788]]